
# Model Training & Evaluation Pipeline with LoRA + 4-bit Quantization

This repository contains a full pipeline for **multi-class text classification** using a transformer model (e.g., LLaMA 3.1 8B by default) with **LoRA (parameter-efficient fine-tuning)** and **4-bit quantization** (BitsAndBytes). It implements a **cyclic 5-fold scheme** (validation = fold *r*, test = fold *(r+1) % 5*), early stopping on validation **loss**, best-model selection by **Macro\_F1**, and optional **temperature scaling** saved per fold and for the final model.

## Overview

1. **Data Loading & Preprocessing**

   * Loads a CSV with columns `comment` and `labels`.
   * Maps string labels to **integer class IDs** and logs a **class distribution** summary.

2. **Model & Tokenizer**

   * Loads a pretrained model with **4-bit quantization** via `BitsAndBytesConfig` (`nf4`, double quant, `bfloat16` compute).
   * Configures padding token (uses EOS as PAD to avoid padding errors).

3. **LoRA Integration**

   * Applies LoRA adapters (targets `q_proj`, `k_proj`, `v_proj`; in test mode only `q_proj`, `v_proj`) and prepares the model for k-bit training.

4. **Cyclic Cross-Validation (5 folds)**

   * For each round **r**:

     * **val = fold r**, **test = fold (r+1) % 5**, **train** = the remaining 3 folds (union of their original `test_idx`).
     * Trains with **early stopping on validation loss** (patience = 3).
     * Selects best checkpoint by **Macro\_F1** (even though early stop is driven by loss).
     * Evaluates on the **test** split and appends metrics to `Performance_Detailed.csv`.
   * After all rounds, appends a **“Mean±95%”** row (mean ± 95% CI across folds).

5. **Final Model Training**

   * Performs a fresh **stratified split** (default 80% train / 20% val).
   * Retrains the final model, evaluates and appends a **Final-Model** row to the CSV, and saves the model and tokenizer to `./saved_model`.

6. **Temperature Scaling (Optional, Included)**

   * Fits a scalar temperature **T** on the validation set by minimizing NLL (LBFGS over `logT`).
   * Saves `temperature.json` per fold (e.g., `./results/fold_r0/temperature.json`) and for the **final model** (`./saved_model/temperature.json`).

## Key Features

* **LoRA** for efficient fine-tuning on large models.
* **4-bit quantization** with `nf4` + double quantization, compute in `bfloat16`.
* **Class-weighted loss** (inverse frequency, normalized) with **label smoothing** (0.05).
* **Cyclic 5-fold CV** where each fold is used once as **validation** and once as **test**.
* **Early stopping** on validation **loss** (patience=3, `min_delta=1e-4`).
* **Best model selection** by **Macro\_F1** at epoch level.
* **Cosine LR schedule**, `warmup_ratio=0.05`, gradient accumulation = 2.
* Mixed precision: **fp16** (training) and **tf32** enabled.
* Results appended to **`Performance_Detailed.csv`** with:

  * One row per round (`fold=r0..r4`)
  * A **“Mean±95%”** summary row
  * A **“Final-Model”** row
* **Batch inference** on evaluation to reduce memory spikes.

## Metrics Logged

For each evaluation, the pipeline records:

* **Accuracy**, **Balanced Accuracy**
* **Per-class** Precision / Recall / F1
* **Macro** Precision / Recall / F1
  (Per-fold rows use the scikit-learn `classification_report` dictionary; the CSV flattens those keys.)

## Training Details (as implemented)

* **Tokenizer max length**: `MAX_LENGTH = 128` (96 in test mode).
* **Batch size**: `BATCH_SIZE = 128` (1 in test mode).
* **Epochs**: up to `N_EPOCHS = 50` (3 in test mode) with early stopping.
* **Optimizer**: `adamw_torch`, `weight_decay=0.01`, `max_grad_norm=1.0`.
* **DataLoader workers**: `dataloader_num_workers=16`.
* **Gradient checkpointing**: enabled to save memory.
* **Random seeds**: multiple sources seeded (`numpy`, `random`, `torch`, CUDA). CuDNN set to deterministic.

## Reproducibility

* Seeds are fixed (`SEED=1991`) and CuDNN is deterministic.
* `CUDA_LAUNCH_BLOCKING=1` aids debugging; memory config via `PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:32`.
* `WANDB_DISABLED=true` and `TOKENIZERS_PARALLELISM=false` to reduce noise/warnings.

## Inputs & Outputs

* **Input CSV**: `FinalLabels.csv` with columns:

  * `comment` (text)
  * `labels` (string label; mapped to int)
* **Outputs**:

  * `./results/fold_r*/` — checkpoints + `temperature.json` per round.
  * `Performance_Detailed.csv` — per-round metrics, **Mean±95%**, and **Final-Model**.
  * `./saved_model/` — final model, tokenizer, and `temperature.json`.

## Requirements

* `transformers`, `datasets`, `peft`, `scikit-learn`, `torch`, `pandas`, `tqdm`, `scipy`, `bitsandbytes` (via transformers’ quantization).

> Tip: Ensure CUDA-compatible builds of PyTorch and BitsAndBytes for 4-bit.

## Usage (high level)

1. Place `FinalLabels.csv` in the working directory with `comment` and `labels`.
2. (Optional) Set `FLAG_TEST=True` for a fast dry run (smaller model, fewer folds/epochs).
3. Run the script. It will:

   * Check CUDA, load data, map labels.
   * Run **cyclic 5-fold CV** with LoRA + 4-bit.
   * Append metrics to `Performance_Detailed.csv` (including **Mean±95%**).
   * Retrain and save the **final model** to `./saved_model`.

## Notes & Caveats

* **Token management**: the code uses a hardcoded `hf_token`. For security, consider moving it to an environment variable (e.g., `HF_TOKEN`) and passing it to `from_pretrained(..., token=os.getenv("HF_TOKEN"))`.
* **Hardware**: 4-bit quantization reduces memory, but you still need a GPU with sufficient VRAM for the chosen model (`meta-llama/Meta-Llama-3.1-8B` by default). Use `FLAG_TEST=True` to verify the pipeline quickly with `TinyLlama`.
* **Validation vs. Selection**: Early stopping is based on **loss** (smoother signal), while the best checkpoint for saving is chosen by **Macro\_F1**—this is intentional to balance stability and task objective.



In [ ]:
# -*- coding: utf-8 -*-
"""
Treinador multilabel (2 rótulos binários):
- Colunas de entrada: 'comment', 'Autorrelato', 'Sintomas / Efeitos Colaterais'
- Vetor multilabel por amostra: [Autorrelato, Sintomas/Efeitos]
- Treinador com BCEWithLogits + pos_weight
- CV estratificada multilabel (MSKF) no esquema cíclico (val=r, test=r+1)
- LoRA sobre modelo base quantizado em 4-bit
- Métricas micro/macro + por rótulo com nomes legíveis
- Tuning de limiar por rótulo via F1 na validação (salvo e reutilizado)
"""

# --- Built-ins
import os
import gc
import logging
import warnings
import json
import random
import shutil

# --- Terceiros: numéricos e utilitários
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats

# --- PyTorch 
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.optim as optim
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

import torchvision
torchvision.disable_beta_transforms_warning()

from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from sklearn.metrics import precision_recall_fscore_support, precision_recall_curve

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    TrainingArguments, Trainer, default_data_collator, TrainerCallback,
    EarlyStoppingCallback, EvalPrediction, set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --------------------------------------------------------------------
# Configs/ajustes gerais
# --------------------------------------------------------------------
torchvision.disable_beta_transforms_warning()
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Evita warning do tokenizer
os.environ["WANDB_DISABLED"] = "true"          # Desabilita W&B
os.environ['CUDA_LAUNCH_BLOCKING'] = "0"       # Debug CUDA melhor
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"

warnings.filterwarnings(
    "ignore",
    message="`torch.cpu.amp.autocast.*` is deprecated",
    category=FutureWarning
)

# Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("multilabel-trainer")

# Token Hugging Face (use variável de ambiente; não hardcode!)
hf_token = os.getenv("HF_TOKEN", None)

# Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------------------------
# Constantes de Treino
# --------------------------------------------------------------------
FLAG_TEST = False              # Se True, reduz dados/épocas para debug
MAX_LENGTH = 96 if FLAG_TEST else 128
SEED = 1991
N_EPOCHS = 3 if FLAG_TEST else 50
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1 if FLAG_TEST else 12
THRESH = 0.5  # limiar global default (trocado pelos por-rótulo se existirem)

# --------------------------------------------------------------------
# Seeds & workers
# --------------------------------------------------------------------
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
random.seed(SEED)
set_seed(SEED)

def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

# --------------------------------------------------------------------
# Utilitários de dados
# --------------------------------------------------------------------
def load_data(file_path: str) -> pd.DataFrame:
    """
    Lê CSV com colunas:
      - 'comment'
      - 'Autorrelato' (0/1)
      - 'Sintomas / Efeitos Colaterais' (0/1)
    Constrói 'labels' como listas [Autorrelato, Sintomas].
    """
    logger.info(f"Loading data from file: {file_path}")
    df = pd.read_csv(file_path, usecols=['comment', 'Autorrelato', 'Sintomas / Efeitos Colaterais'])
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    if FLAG_TEST:
        df = df.sample(100, random_state=42)

    df['Autorrelato'] = df['Autorrelato'].astype(int).clip(0, 1)
    df['Sintomas / Efeitos Colaterais'] = df['Sintomas / Efeitos Colaterais'].astype(int).clip(0, 1)

    df['labels'] = df.apply(
        lambda r: [int(r['Autorrelato']), int(r['Sintomas / Efeitos Colaterais'])],
        axis=1
    )
    logger.info(f"Data loaded. Total samples: {len(df)}")
    return df

def cleanup_checkpoints(path: str):
    """Remove diretório de checkpoints para economizar espaço."""
    try:
        shutil.rmtree(path, ignore_errors=True)
        logger.info(f"Checkpoints removed at: {path}")
    except Exception as e:
        logger.warning(f"Failed to remove {path}: {e}")

def map_labels(df: pd.DataFrame):
    """
    Valida o shape do vetor multilabel e retorna:
      - df (cópia)
      - num_labels
      - label_names (nomes legíveis na ordem do vetor)
    """
    df = df.copy()
    Y = np.vstack(df['labels'].values).astype(int)
    assert Y.shape[1] == 2, f"Esperado 2 rótulos, obtido {Y.shape[1]}"
    label_names = ['Autorrelato', 'Sintomas / Efeitos Colaterais']
    pos_counts = Y.sum(axis=0)
    for i, lab in enumerate(label_names):
        logger.info(f"Label '{lab}' (idx {i}) - positivos: {int(pos_counts[i])}")
    return df, Y.shape[1], label_names

# --------------------------------------------------------------------
# Threshold tuning utilitários
# --------------------------------------------------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def tune_thresholds_by_f1(y_true: np.ndarray, y_prob: np.ndarray,
                          floor: float = 0.05, ceil: float = 0.95) -> np.ndarray:
    """
    Define um limiar por rótulo maximizando F1 na validação.
    """
    L = y_true.shape[1]
    ths = np.full(L, 0.5, dtype=float)
    for j in range(L):
        yj = y_true[:, j]
        pj = y_prob[:, j]
        if yj.sum() == 0:
            ths[j] = 0.5
            continue
        p, r, t = precision_recall_curve(yj, pj)
        f1 = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
        if len(t) == 0:
            ths[j] = 0.5
            continue
        f1_aligned = f1[1:]
        best_idx = int(np.argmax(f1_aligned))
        th_j = float(t[best_idx])
        ths[j] = float(np.clip(th_j, floor, ceil))
    return ths

# def compute_metrics_per_class(p, label_names=None, threshold=0.5):
#     """
#     Retorna métricas micro/macro e por classe.
#     - p.predictions: logits [N, L]
#     - p.label_ids:   multi-hot [N, L]
#     """
#     logits = p.predictions
#     y_true = p.label_ids

#     # Sigmoid -> probabilidades -> binariza pelo threshold
#     y_prob = 1.0 / (1.0 + np.exp(-logits))
#     y_pred = (y_prob >= threshold).astype(int)

#     # Agregadas
#     prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
#         y_true, y_pred, average='micro', zero_division=0
#     )
#     prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
#         y_true, y_pred, average='macro', zero_division=0
#     )

#     # Por classe
#     prec_none, rec_none, f1_none, _ = precision_recall_fscore_support(
#         y_true, y_pred, average=None, zero_division=0
#     )

#     # Monta o dicionário de retorno (Trainer loga tudo isso)
#     metrics = {
#         "micro_precision": float(prec_micro),
#         "micro_recall":    float(rec_micro),
#         "micro_f1":        float(f1_micro),
#         "macro_precision": float(prec_macro),
#         "macro_recall":    float(rec_macro),
#         "macro_f1":        float(f1_macro),
#     }

#     # Nomes legíveis para cada classe
#     if label_names is None:
#         label_names = [f"label_{i}" for i in range(len(prec_none))]

#     for name, p_, r_, f_ in zip(label_names, prec_none, rec_none, f1_none):
#         # use nomes curtos nas chaves para o log ficar compacto
#         key = name.replace(" ", "_").replace("/", "_")
#         metrics[f"{key}_precision"] = float(p_)
#         metrics[f"{key}_recall"]    = float(r_)
#         metrics[f"{key}_f1"]        = float(f_)

#     return metrics

    
def save_thresholds(path: str, thresholds: np.ndarray) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"thresholds": thresholds.tolist()}, f)

def load_thresholds(path: str) -> np.ndarray:
    with open(path, "r", encoding="utf-8") as f:
        return np.array(json.load(f)["thresholds"], dtype=float)

def _collect_logits_and_labels(model, eval_dataset, batch_size: int = BATCH_SIZE):
    """
    Coleta logits crus e labels de um Dataset HF via DataLoader.
    Retorna:
      - logits: [N, L]
      - labels: [N, L]
    """
    model.eval()
    dl = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, collate_fn=default_data_collator)
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in dl:
            labels = batch["labels"]
            if isinstance(labels, torch.Tensor):
                labels_np = labels.numpy()
            else:
                labels_np = np.array(labels)
            labels_list.append(labels_np)

            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            out = model(**batch)
            logits_list.append(out.logits.detach().cpu().numpy())
    logits = np.concatenate(logits_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    return logits, labels

# --------------------------------------------------------------------
# Modelo & LoRA
# --------------------------------------------------------------------
def load_model(base_model_name: str, num_labels: int):
    """
    Carrega modelo 4-bit + tokenizer.
    """
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    logger.info(f"Loading model {base_model_name} with {num_labels} labels (multilabel).")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=hf_token, add_prefix_space=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        token=hf_token,
        quantization_config=quantization_config,
        num_labels=num_labels
    )
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.config.problem_type = "multi_label_classification"
    return model, tokenizer

def apply_lora(model):
    """
    Aplica LoRA para baratear o fine-tuning.
    """
    logger.info("Applying LoRA to the model.")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'] if not FLAG_TEST else ['q_proj', 'v_proj'],
        task_type="SEQ_CLS",
        bias="none",
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    return model

# --------------------------------------------------------------------
# Pesos de classe e métricas
# --------------------------------------------------------------------
def compute_class_weights(y_train_multi):
    """
    pos_weight por rótulo (para BCEWithLogitsLoss): (N - pos)/pos
    """
    Y = np.vstack(y_train_multi)  # [N, L]
    N = Y.shape[0]
    pos = Y.sum(axis=0)
    pos = np.clip(pos, 1.0, None)
    neg = N - pos
    pos_weight = torch.tensor(neg / pos, dtype=torch.float32, device=device)  # [L]
    return pos_weight

def compute_metrics_per_class(p, label_names=None, threshold=0.5, log_tuned=True):
    logits = p.predictions
    y_true = p.label_ids
    y_prob = 1.0 / (1.0 + np.exp(-logits))

    # --- Métricas padrão (threshold fixo) ---
    y_pred = (y_prob >= threshold).astype(int)
    prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='micro', zero_division=0
    )
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    metrics = {
        "micro_precision": float(prec_micro),
        "micro_recall":    float(rec_micro),
        "micro_f1":        float(f1_micro),
        "macro_precision": float(prec_macro),
        "macro_recall":    float(rec_macro),
        "macro_f1":        float(f1_macro),
    }

    # nomes legíveis por classe
    if label_names is None:
        label_names = [f"label_{i}" for i in range(y_true.shape[1])]
    p_none, r_none, f_none, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    for name, p_, r_, f_ in zip(label_names, p_none, r_none, f_none):
        key = name.replace(" ", "_").replace("/", "_")
        metrics[f"{key}_precision"] = float(p_)
        metrics[f"{key}_recall"]    = float(r_)
        metrics[f"{key}_f1"]        = float(f_)

    # --- EXTRA: métricas com thresholds tunados na própria validação ---
    if log_tuned:
        ths = tune_thresholds_by_f1(y_true, y_prob)              # vetor [L]
        y_pred_t = (y_prob >= ths.reshape(1, -1)).astype(int)

        pm_t, rm_t, fm_t, _ = precision_recall_fscore_support(
            y_true, y_pred_t, average='micro', zero_division=0
        )
        Pa_t, Ra_t, Fa_t, _ = precision_recall_fscore_support(
            y_true, y_pred_t, average='macro', zero_division=0
        )
        metrics.update({
            "micro_f1_tuned": float(fm_t),
            "macro_f1_tuned": float(Fa_t),
            "micro_precision_tuned": float(pm_t),
            "micro_recall_tuned":    float(rm_t),
            "macro_precision_tuned": float(Pa_t),
            "macro_recall_tuned":    float(Ra_t),
        })

        p_t, r_t, f_t, _ = precision_recall_fscore_support(
            y_true, y_pred_t, average=None, zero_division=0
        )
        for name, p_, r_, f_ in zip(label_names, p_t, r_t, f_t):
            key = name.replace(" ", "_").replace("/", "_")
            metrics[f"{key}_precision_tuned"] = float(p_)
            metrics[f"{key}_recall_tuned"]    = float(r_)
            metrics[f"{key}_f1_tuned"]        = float(f_)

    return metrics

# --------------------------------------------------------------------
# Agregação de métricas (CSV)
# --------------------------------------------------------------------
def calculate_mean_and_confidence_interval_from_csv(file_name):
    """
    Calcula média e IC95% por métrica (linhas de folds) e anexa uma linha 'Mean±95%'.
    """
    df = pd.read_csv(file_name)
    df_folds = df[(df['fold'] != 'Mean±95%') & (df['fold'] != 'Final-Model')]
    df_numeric = df_folds.drop(columns=['fold', 'epoch'], errors='ignore')

    mean_metrics = df_numeric.mean(numeric_only=True)
    std_error = df_numeric.sem(numeric_only=True)
    ci95 = std_error * stats.t.ppf((1 + 0.95) / 2., len(df_numeric) - 1)

    mean_row = {'fold': 'Mean±95%', 'epoch': ''}
    for col in mean_metrics.index:
        mean_row[col] = f"{mean_metrics[col]:.4f} ± {ci95[col]:.4f}"
    pd.DataFrame([mean_row]).to_csv(file_name, mode='a', header=False, index=False)

def add_final_model_metrics(file_name, metrics, num_labels):
    rows = {
        'fold': 'Final-Model',
        'epoch': '',
        'micro_precision': f"{metrics.get('micro_precision', 0.0):.3f}",
        'micro_recall':    f"{metrics.get('micro_recall',    0.0):.3f}",
        'micro_f1':        f"{metrics.get('micro_f1',        0.0):.3f}",
        'macro_precision': f"{metrics.get('macro_precision', 0.0):.3f}",
        'macro_recall':    f"{metrics.get('macro_recall',    0.0):.3f}",
        'macro_f1':        f"{metrics.get('macro_f1',        0.0):.3f}",
    }
    # mantém compatibilidade com leitor antigo (indices numéricos)
    for i in range(num_labels):
        keyp = f"label_{i}_precision"
        keyr = f"label_{i}_recall"
        keyf = f"label_{i}_f1"
        if keyp in metrics:
            rows[keyp] = f"{metrics[keyp]:.3f}"
        if keyr in metrics:
            rows[keyr] = f"{metrics[keyr]:.3f}"
        if keyf in metrics:
            rows[keyf] = f"{metrics[keyf]:.3f}"
    pd.DataFrame([rows]).to_csv(file_name, mode='a', header=False, index=False)

class WeightedTrainer(Trainer):
    """
    Trainer customizado usando BCEWithLogitsLoss com pos_weight para balancear classes.
    """
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels").float()
        outputs = model(**{k:v for k,v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def append_metrics_to_csv(file_name, metrics, fold_id, epoch):
    logger.info(f"Saving metrics for fold {fold_id}, epoch {int(epoch)}.")
    rows = {
        'fold': fold_id,
        'epoch': int(epoch)-EARLY_STOPPING_PATIENCE,
        'micro_precision': f"{metrics.get('micro_precision', 0.0):.3f}",
        'micro_recall':    f"{metrics.get('micro_recall',    0.0):.3f}",
        'micro_f1':        f"{metrics.get('micro_f1',        0.0):.3f}",
        'macro_precision': f"{metrics.get('macro_precision', 0.0):.3f}",
        'macro_recall':    f"{metrics.get('macro_recall',    0.0):.3f}",
        'macro_f1':        f"{metrics.get('macro_f1',        0.0):.3f}",
    }
    # compat com leitores existentes (indices numéricos)
    i = 0
    while True:
        kp, kr, kf = f"label_{i}_precision", f"label_{i}_recall", f"label_{i}_f1"
        if kp in metrics or kr in metrics or kf in metrics:
            rows[kp] = f"{metrics.get(kp, 0.0):.3f}"
            rows[kr] = f"{metrics.get(kr, 0.0):.3f}"
            rows[kf] = f"{metrics.get(kf, 0.0):.3f}"
            i += 1
        else:
            break

    df = pd.DataFrame([rows])
    if os.path.exists(file_name):
        df.to_csv(file_name, mode='a', header=False, index=False)
    else:
        df.to_csv(file_name, index=False)

class LossEarlyStoppingCallback(TrainerCallback):
    """
    Early stopping pela perda de validação (mais suave que F1).
    """
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.count = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics:
            return
        current = metrics.get("eval_loss")
        if current is None:
            return
        if self.best is None or (self.best - current) > self.min_delta:
            self.best = current
            self.count = 0
        else:
            self.count += 1
            if self.count >= self.patience:
                control.should_training_stop = True
        return control

# --------------------------------------------------------------------
# Loop de CV + treino final
# --------------------------------------------------------------------
def evaluate_model_on_test_set(model, tokenizer, test_df, batch_size=BATCH_SIZE, thresholds=None):
    """
    Roda inferência batelada no conjunto de teste.
    test_df: colunas 'text' e 'labels'
    """
    sentences = test_df['text'].tolist()
    all_logits = []
    logger.info("Starting test set evaluation (multilabel).")
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch_sentences = sentences[i:i + batch_size]
        inputs = tokenizer(batch_sentences, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            all_logits.append(outputs['logits'].to('cpu'))
            del outputs
            torch.cuda.empty_cache()
    logits = torch.cat(all_logits, dim=0).numpy()
    probs = _sigmoid(logits)

    if thresholds is None:
        preds = (probs >= THRESH).astype(int)
    else:
        thr = np.asarray(thresholds, dtype=float).reshape(1, -1)
        preds = (probs >= thr).astype(int)

    test_df = test_df.copy()
    test_df['predictions'] = list(preds)
    return test_df

def get_metrics_result(test_df: pd.DataFrame):
    """
    Calcula métricas padrão a partir das previsões binárias.
    """
    y_true = np.vstack(test_df['labels'].values)
    y_pred = np.vstack(test_df['predictions'].values)
    prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='micro', zero_division=0
    )
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    prec_none, rec_none, f1_none, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    metrics = {
        "micro_precision": prec_micro,
        "micro_recall": rec_micro,
        "micro_f1": f1_micro,
        "macro_precision": prec_macro,
        "macro_recall": rec_macro,
        "macro_f1": f1_macro,
    }
    for i, (p_, r_, f_) in enumerate(zip(prec_none, rec_none, f1_none)):
        metrics[f"label_{i}_precision"] = p_
        metrics[f"label_{i}_recall"]    = r_
        metrics[f"label_{i}_f1"]        = f_
    return metrics

def make_training_args_compat(**kwargs):
    """
    Cria TrainingArguments compatível com a versão instalada do transformers:
    - filtra kwargs não suportados
    - faz fallback evaluation_strategy -> eval_strategy se necessário
    """
    import inspect
    from transformers import TrainingArguments

    sig = inspect.signature(TrainingArguments.__init__)
    params = set(sig.parameters.keys())

    # Fallback de nome do parâmetro se 'evaluation_strategy' não existir mas 'eval_strategy' existir
    if "evaluation_strategy" not in params and "eval_strategy" in params:
        if "evaluation_strategy" in kwargs and "eval_strategy" not in kwargs:
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")

    # Filtra apenas os kwargs suportados pela versão atual
    filtered = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**filtered)

def fit_temperature(model, eval_dataset, device, batch_size):
    """
    ⚠️ Normalmente evitamos temperature scaling em multilabel com BCE.
    Mantido aqui apenas como exemplo, não é usado no fluxo principal.
    """
    model.eval()
    logits_list, labels_list = [], []
    val_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, collate_fn=default_data_collator)
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            outputs = model(**batch)
            logits_list.append(outputs.logits)
            labels_list.append(labels)
    logits = torch.cat(logits_list, dim=0).to(device)
    labels = torch.cat(labels_list, dim=0).to(device)

    logT = nn.Parameter(torch.zeros(1, device=device))
    nll = nn.CrossEntropyLoss()
    optimizer = optim.LBFGS([logT], lr=0.01, max_iter=50, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad(set_to_none=True)
        T = torch.exp(logT)
        loss = nll(logits / T, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    T_value = torch.exp(logT).detach().item()
    return T_value

def train(model, tokenizer, train_data, eval_data, y_train, y_eval, fold_id,
          trainer_epochs=N_EPOCHS, label_names=None):
    """
    Treina o modelo com dados de treino/validação.
    """
    train_df = pd.DataFrame({'text': train_data, 'labels': list(y_train)})
    eval_df  = pd.DataFrame({'text': eval_data,  'labels': list(y_eval)})

    def tokenize_function(examples):
        return tokenizer(examples['text'], padding=True, truncation=True, max_length=MAX_LENGTH)

    train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
    eval_dataset  = Dataset.from_pandas(eval_df ).map(tokenize_function, batched=True)

    def set_labels(batch):
        batch['labels'] = [torch.tensor(x, dtype=torch.float32) for x in batch['labels']]
        return batch

    train_dataset = train_dataset.map(set_labels, batched=True)
    eval_dataset  = eval_dataset.map(set_labels,  batched=True)

    train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

    class_weights = compute_class_weights(y_train)

    if hasattr(model, 'config'):
        if label_names is None:
            label_names = [str(i) for i in range(model.config.num_labels)]
        model.config.label2id = {name: i for i, name in enumerate(label_names)}
        model.config.id2label = {i: name for i, name in enumerate(label_names)}

    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    training_args = make_training_args_compat(
        output_dir=f'./results/fold_{fold_id}',
        num_train_epochs=trainer_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        gradient_accumulation_steps=2,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_strategy="epoch",          # LOG a cada epoch
        save_strategy='epoch',
        evaluation_strategy="epoch",       # AVALIA a cada epoch
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",  # Para salvar o melhor
        greater_is_better=True,
        gradient_checkpointing=False,
        fp16 = not use_bf16,
        bf16 = use_bf16,
        report_to="none",
        save_total_limit=EARLY_STOPPING_PATIENCE,
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim="adamw_torch",
        dataloader_num_workers=16,
    )
    
    def _compute_metrics(p):
        return compute_metrics_per_class(p, label_names=label_names, threshold=THRESH)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,  
        compute_metrics=_compute_metrics,
        callbacks=[LossEarlyStoppingCallback(patience=EARLY_STOPPING_PATIENCE, min_delta=1e-4)],
        class_weights=class_weights,
    )

    # Algumas versões não aceitam label_names no __init__. Apenas um teste
    if hasattr(trainer, "label_names"):
        trainer.label_names = ["labels"]

    torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker)

    trainer.train()
    trainer.evaluate(eval_dataset=eval_dataset)

    # Tuning de limiares por rótulo na validação
    try:
        val_logits, val_labels = _collect_logits_and_labels(trainer.model, eval_dataset, batch_size=BATCH_SIZE)
        val_probs = _sigmoid(val_logits)
        tuned_thresholds = tune_thresholds_by_f1(val_labels, val_probs)
        save_thresholds(f'./results/fold_{fold_id}/thresholds.json', tuned_thresholds)
        logger.info(f"Per-label thresholds saved to ./results/fold_{fold_id}/thresholds.json")
    except Exception as e:
        logger.warning(f"Failed to tune/save thresholds for fold {fold_id}: {e}")

    return trainer

def retrain_final_model(df, num_labels, model_name, tokenizer, test_size=0.2, label_names=None):
    """
    Re-treina modelo final em (1-test_size) e valida em test_size (MSKF 1 split).
    Salva modelo+tokenizer+limiares e escreve métricas finais no CSV.
    """
    logger.info(f"Performing new iterative stratification split: {(1 - test_size)*100:.0f}% train / {test_size*100:.0f}% validation")

    Y_all = np.vstack(df['labels'].values).astype(int)
    mskf = MultilabelStratifiedKFold(n_splits=int(1/(test_size)), shuffle=True, random_state=SEED)
    train_idx, val_idx = next(mskf.split(np.zeros(len(df)), Y_all))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    X_train, y_train = train_df['comment'], train_df['labels']
    X_val,   y_val   = val_df['comment'],  val_df['labels']

    model, tokenizer = load_model(model_name, num_labels)
    model = apply_lora(model)

    trainer = train(model, tokenizer, X_train, X_val, y_train, y_val, fold_id="Final", label_names=label_names)

    tuned_thresholds = None
    thr_path = './results/fold_Final/thresholds.json'
    if os.path.exists(thr_path):
        try:
            tuned_thresholds = load_thresholds(thr_path)
            logger.info(f"Loaded tuned thresholds for final model from {thr_path}")
        except Exception as e:
            logger.warning(f"Could not load thresholds from {thr_path}: {e}")

    val_df = pd.DataFrame({'text': X_val, 'labels': y_val})
    val_df = evaluate_model_on_test_set(model, tokenizer, val_df, thresholds=tuned_thresholds)
    final_metrics = get_metrics_result(val_df)
    add_final_model_metrics('Performance_Detailed.csv', final_metrics, num_labels=num_labels)

    model.save_pretrained('./saved_model')
    tokenizer.save_pretrained('./saved_model')
    if tuned_thresholds is not None:
        save_thresholds('./saved_model/thresholds.json', tuned_thresholds)

    logger.info("Final model saved in ./saved_model")
    cleanup_checkpoints('./results/fold_Final')

def run_cross_validation(df, num_labels, model_name, tokenizer, label_names):
    """
    CV cíclica para multilabel:
      round r: val=r, test=(r+1)%k, train = união dos demais.
    Cada fold atua uma vez como validação e uma como teste.
    """
    logger.info("Starting cyclic CV (multilabel, val=r, test=r+1).")
    csv_path = 'Performance_Detailed.csv'
    folds_n = 2 if FLAG_TEST else 5

    completed_rounds = set()
    if os.path.exists(csv_path):
        df_csv = pd.read_csv(csv_path)
        df_csv['fold'] = df_csv['fold'].astype(str)
        completed_rounds = set([f for f in df_csv['fold'].unique() if f.startswith("r")])
        logger.info(f"Already processed rounds: {sorted(completed_rounds)}")
        if all(f"r{i}" in completed_rounds for i in range(folds_n)) and 'Mean±95%' in set(df_csv['fold'].unique()):
            logger.info("All rounds already processed. Skipping CV and starting final training.")
            retrain_final_model(df, num_labels, model_name, tokenizer, label_names=label_names)
            return

    Y_all = np.vstack(df['labels'].values).astype(int)
    mskf = MultilabelStratifiedKFold(n_splits=folds_n, shuffle=True, random_state=SEED)
    fold_indices = list(mskf.split(np.zeros(len(df)), Y_all))

    epoch_list = []

    for r in range(folds_n):
        fold_tag = f"r{r}"
        if fold_tag in completed_rounds:
            logger.info(f"Round {fold_tag} has already been processed. Skipping...")
            continue

        val_fold  = r
        test_fold = (r + 1) % folds_n

        train_idx = np.concatenate([
            idx for k, (_, idx) in enumerate(fold_indices)
            if k not in [val_fold, test_fold]
        ])
        val_idx  = fold_indices[val_fold][1]
        test_idx = fold_indices[test_fold][1]

        tr = df.iloc[train_idx].reset_index(drop=True)
        va = df.iloc[val_idx].reset_index(drop=True)
        te = df.iloc[test_idx].reset_index(drop=True)

        X_train, y_train = tr['comment'], tr['labels']
        X_val,   y_val   = va['comment'], va['labels']
        X_test,  y_test  = te['comment'], te['labels']

        model, tok = load_model(model_name, num_labels)
        model = apply_lora(model)
        
        # Tenta usar FlashAttention 2 se o backend suportar
        try:
            model.config.attn_implementation = "flash_attention_2"
        except Exception as e:
            print(f"FlashAttention 2 não disponível: {e}")
        
        trainer = train(
            model, tok, 
            X_train, X_val, 
            y_train, y_val, 
            fold_id=fold_tag,
            trainer_epochs=N_EPOCHS, 
            label_names=label_names
        )

        epoch_list.append(trainer.state.epoch)

        tuned_thresholds = None
        thr_path = f'./results/fold_{fold_tag}/thresholds.json'
        if os.path.exists(thr_path):
            try:
                tuned_thresholds = load_thresholds(thr_path)
                logger.info(f"Loaded tuned thresholds from {thr_path}")
            except Exception as e:
                logger.warning(f"Could not load thresholds from {thr_path}: {e}")

        test_df = pd.DataFrame({'text': X_test, 'labels': y_test})
        test_df = evaluate_model_on_test_set(model, tok, test_df, thresholds=tuned_thresholds)
        metrics = get_metrics_result(test_df)

        append_metrics_to_csv(csv_path, metrics, fold_tag, trainer.state.epoch)

        del model, tok, trainer
        torch.cuda.empty_cache()
        gc.collect()
        cleanup_checkpoints(f'./results/fold_{fold_tag}')

    if len(epoch_list) == 0:
        logger.info("No new round was executed in this run.")
        return

    calculate_mean_and_confidence_interval_from_csv(csv_path)
    retrain_final_model(df, num_labels, model_name, tokenizer, label_names=label_names)

# --------------------------------------------------------------------
# Utilitário de diagnóstico
# --------------------------------------------------------------------
def check_cuda():
    print("CUDA available?", torch.cuda.is_available())
    print("Number of CUDA GPUs:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
        print("Total GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
        print("Available GPU memory (GB):", round(torch.cuda.memory_reserved(0) / 1e9, 2))
    else:
        print("CUDA is not available. Please check driver and CUDA toolkit installation.")

# --------------------------------------------------------------------
# Main
# --------------------------------------------------------------------
if __name__ == "__main__":
    check_cuda()
    WORKSPACE = "./"
    file_path = f"{WORKSPACE}FinalLabels.csv"  # ajuste se necessário

    df = load_data(file_path)
    df, num_labels, label_names = map_labels(df)

    Y_tmp = np.vstack(df['labels'].values)
    logger.info(f"Multilabel matrix shape: {Y_tmp.shape}, mean positivity: {Y_tmp.mean():.6f}")

    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" if FLAG_TEST else "meta-llama/Meta-Llama-3.1-8B"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token, add_prefix_space=True)

    run_cross_validation(df, num_labels, MODEL_NAME, tokenizer, label_names=label_names)


# Teste
Este script executa inferência de um classificador multilabel com dois rótulos
— ["Autorrelato", "Sintomas / Efeitos Colaterais"] — usando:
- Modelo em 4-bit (NF4) + FP16 (evita erros de BFloat16)
- Suporte a PEFT/LoRA (se houver adapter_config.json) ou modelo final mesclado
- Sigmoid + limiar por rótulo (lido de ./saved_model/thresholds.json; fallback 0.5)

Há dois modos de uso, já implementados no mesmo arquivo:

1) MODO EXEMPLO (pequenas listas em memória)
   - Função: `classify_and_save(...)`
   - Entrada: lista de strings (ex.: `examples = [...]`)
   - Saída: CSV com uma linha por texto + 4 colunas novas de predição:
       probs_Autorrelato, pred_Autorrelato,
       probs_Sintomas_Efeitos_Colaterais, pred_Sintomas_Efeitos_Colaterais
   - Útil para smoke tests rápidos, debugging e validações pontuais.

2) MODO BASE REAL EM CSV (grande volume, com chunks, tqdm e retomada)
   - Função/entrypoint: `classify_file(...)`
   - Lê `prediction/final_comments_match_cleaned.csv` em CHUNKS (padrão: 100k linhas)
   - Coluna de texto obrigatória: `comment` (configurável via TEXT_COL)
   - Mostra barra de progresso com ETA (tqdm) e contabiliza linhas previamente
   - Após cada chunk:
       - Anexa os resultados em `prediction/final_comments_match_cleaned_pred.csv`
       - Atualiza um arquivo de estado `*.state.json` com o "próximo chunk"
         → Em caso de queda de energia, ao relançar o script com os mesmos caminhos,
           ele RETOMA exatamente de onde parou, sem duplicar linhas.
   - Mantém TODAS as colunas originais do CSV e adiciona exatamente 4 colunas:
       probs_Autorrelato, pred_Autorrelato,
       probs_Sintomas_Efeitos_Colaterais, pred_Sintomas_Efeitos_Colaterais

Detalhes de implementação
-------------------------
- Quantização: BitsAndBytes (NF4) em 4-bit + `torch.float16` (compute dtype)
- Padding garantido: ajusta `pad_token` e `pad_token_id` (Llama-friendly para batch>1)
- Thresholds:
    - Carrega de `./saved_model/thresholds.json` (chave "thresholds", vetor por rótulo)
    - Se o arquivo não existir ou for inválido, usa 0.5 por rótulo (fallback)
    - Não há retuning aqui (somente aplicação dos limiares salvos)
- LoRA:
    - Se existir `adapter_config.json` em `MODEL_DIR`, carregamos o base do Hub em 4-bit
      e anexamos o adapter LoRA salvo.
    - Dica: para produção, considere mesclar o LoRA no base (merge_and_unload) e apontar
      `MODEL_DIR = "./saved_model_merged"` para evitar o aviso do classifier head.

Como rodar (rápido)
-------------------
- Teste com exemplos (em memória):
    - Edite a lista `examples` no bloco `if __name__ == "__main__":`
    - Isso gera `predicoes_exemplos.csv`

- Processar a base real:
    - Garante que `prediction/final_comments_match_cleaned.csv` exista
    - Ajuste constantes se necessário (BATCH_SIZE, CHUNK_ROWS, TEXT_COL)
    - Execute o script (ele criará/atualizará `prediction/final_comments_match_cleaned_pred.csv`)
    - Se cair energia, rode novamente: ele continua do último chunk completo,
      conforme `prediction/final_comments_match_cleaned_pred.csv.state.json`.

Colunas adicionadas
-------------------
- `probs_Autorrelato` (float)
- `pred_Autorrelato`  (0/1)
- `probs_Sintomas_Efeitos_Colaterais` (float)
- `pred_Sintomas_Efeitos_Colaterais`  (0/1)

Boas práticas e dicas
---------------------
- Reduza `BATCH_SIZE` e/ou `MAX_LENGTH` se faltar VRAM.
- Para datasets muito grandes, considere usar Parquet (mais rápido e compacto).
- Mantenha `MODEL_DIR` estável entre as execuções (necessário para retomar com consistência).
- Se mesclar o LoRA (modelo final único), a inferência tende a ficar mais simples e sem avisos.
"""


In [1]:
# -*- coding: utf-8 -*-
"""
Multilabel classifier inference (thresholded, no temperature scaling).

- Loads saved model dir (supports PEFT/LoRA adapters) in 4-bit NF4.
- Forces FP16 to avoid 'unsupported BFloat16' errors.
- Ensures padding token is set (fixes batch>1 errors for Llama).
- Predicts probabilities via sigmoid and per-label thresholds (from thresholds.json or 0.5 fallback).
- Works with a list of texts OR a large pandas DataFrame (chunked, memory-friendly).
- Prints human-readable class names and saves results to CSV.

Label names are explicitly set to:
  ["Autorrelato", "Sintomas / Efeitos Colaterais"]
so output columns and prints use these names.

TIP: If you have a merged model (LoRA merged into base, including classifier head),
     set MODEL_DIR to that folder (e.g., './saved_model_merged') to avoid 'score.weight' warnings.
"""

import os
import json
from typing import List, Optional, Sequence, Union

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import PeftModel  # pip install peft

# -----------------------------
# Global settings
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128
BATCH_SIZE = 32
MODEL_DIR = "./saved_model"  # If you have a merged model, point this to './saved_model_merged'

# Desired, human-readable label names in the correct id order.
DESIRED_LABEL_NAMES = ["Autorrelato", "Sintomas / Efeitos Colaterais"]


# -----------------------------
# Helpers
# -----------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    """Vectorized sigmoid for numpy arrays."""
    return 1.0 / (1.0 + np.exp(-x))


def _ensure_padding(tokenizer, model) -> None:
    """
    Ensure we have a pad token and pad_token_id so batches > 1 work with Llama.
    Uses EOS as PAD if no explicit pad token exists.
    """
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tokenizer))
        else:
            tokenizer.pad_token = tokenizer.eos_token

    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    tokenizer.padding_side = "right"


def load_thresholds_or_default(path: str, num_labels: int, default: float = 0.5) -> np.ndarray:
    """
    Load per-label thresholds from a JSON file. If missing or malformed,
    returns a vector filled with `default`.
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            ths = np.array(json.load(f)["thresholds"], dtype=float)
        if ths.shape[0] != num_labels:
            ths = np.full(num_labels, default, dtype=float)
    except Exception:
        ths = np.full(num_labels, default, dtype=float)
    return ths


def load_model_and_tokenizer(model_dir: str = MODEL_DIR):
    """
    Load tokenizer and model in 4-bit (NF4) and attach PEFT/LoRA adapter if present.
    Forces FP16 (safe across GPUs) to avoid 'unsupported BFloat16'.
    Also ensures padding and sets human-readable label names.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

    # Force FP16 compute dtype (robust for environments without bf16 support)
    compute_dtype = torch.float16
    bnb_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    adapter_cfg_path = os.path.join(model_dir, "adapter_config.json")
    if os.path.exists(adapter_cfg_path):
        # PEFT adapter path: load the base in 4-bit and attach adapter
        with open(adapter_cfg_path, "r", encoding="utf-8") as f:
            adapter_cfg = json.load(f)
        base = adapter_cfg.get("base_model_name_or_path")
        if not base:
            raise ValueError("adapter_config.json missing 'base_model_name_or_path'.")

        base_model = AutoModelForSequenceClassification.from_pretrained(
            base,
            quantization_config=bnb_4bit,
            device_map="auto",         # auto placement (offload if needed)
            torch_dtype=compute_dtype, # enforce fp16
        )
        model = PeftModel.from_pretrained(base_model, model_dir)
    else:
        # Full fine-tuned (possibly merged) model saved in this folder: load in 4-bit
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir,
            quantization_config=bnb_4bit,
            device_map="auto",
            torch_dtype=compute_dtype,
        )

    _ensure_padding(tokenizer, model)
    model.eval()

    # Force desired label names (assumes id 0 = Autorrelato, id 1 = Sintomas/Efeitos)
    num_labels = model.config.num_labels
    if len(DESIRED_LABEL_NAMES) != num_labels:
        raise ValueError(
            f"DESIRED_LABEL_NAMES length ({len(DESIRED_LABEL_NAMES)}) "
            f"does not match num_labels ({num_labels})."
        )
    label_names = DESIRED_LABEL_NAMES
    model.config.id2label = {i: name for i, name in enumerate(label_names)}
    model.config.label2id = {name: i for i, name in enumerate(label_names)}

    return model, tokenizer, label_names


# -----------------------------
# Core prediction routines
# -----------------------------
@torch.no_grad()
def predict_multilabel_texts(
    texts: Sequence[str],
    model,
    tokenizer,
    label_names: List[str],
    thresholds: np.ndarray,
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_LENGTH,
) -> pd.DataFrame:
    """
    Predict probabilities and thresholded labels for a list of strings.
    Returns a DataFrame with one row per text, including per-label columns
    based on `label_names`.
    """
    if isinstance(texts, str):
        texts = [texts]

    # Safety: if padding is still missing for some reason, force batch_size=1
    if getattr(model.config, "pad_token_id", None) is None and batch_size > 1:
        batch_size = 1

    probs_list = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length
        )
        inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in inputs.items()}
        logits_t = model(**inputs).logits.detach()       # [B, L]
        probs_t  = torch.sigmoid(logits_t)               # [B, L]
        probs    = probs_t.cpu().numpy().astype("float32")
        probs_list.append(probs)

    probs = np.vstack(probs_list)      # [N, L]
    thr = thresholds.reshape(1, -1)    # [1, L]
    preds = (probs >= thr).astype(int) # [N, L]

    rows = []
    for text, p_row, y_row in zip(texts, probs, preds):
        row = {"text": text}
        for j, name in enumerate(label_names):
            pcol = f"probs_{name}".replace(" ", "_").replace("/", "_")
            ycol = f"pred_{name}".replace(" ", "_").replace("/", "_")
            row[pcol] = float(p_row[j])
            row[ycol] = int(y_row[j])
        rows.append(row)
    return pd.DataFrame(rows)


@torch.no_grad()
def predict_multilabel_dataframe(
    df: pd.DataFrame,
    text_col: str,
    model,
    tokenizer,
    label_names: List[str],
    thresholds: np.ndarray,
    out_csv: Optional[str] = None,
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_LENGTH,
    write_mode: str = "w",
    header: bool = True,
    chunk_rows: int = 100_000,
) -> Optional[pd.DataFrame]:
    """
    Scalable inference for very large DataFrames.
    Processes in row chunks to bound memory, predicts in mini-batches,
    and optionally appends results to CSV.
    """
    assert text_col in df.columns, f"Column '{text_col}' not found."

    collector = [] if out_csv is None else None
    n = len(df)

    for start in range(0, n, chunk_rows):
        end = min(start + chunk_rows, n)
        sub = df.iloc[start:end]
        texts = sub[text_col].astype(str).tolist()

        chunk_rows_out = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            logits_t = model(**inputs).logits.detach()      # tensor
            probs_t  = torch.sigmoid(logits_t)              
            probs    = probs_t.cpu().numpy()               
            thr = thresholds.reshape(1, -1)
            preds = (probs >= thr).astype(int)

            for text, p_row, y_row in zip(batch, probs, preds):
                row = {"text": text}
                for j, name in enumerate(label_names):
                    pcol = f"probs_{name}".replace(" ", "_").replace("/", "_")
                    ycol = f"pred_{name}".replace(" ", "_").replace("/", "_")
                    row[pcol] = float(p_row[j])
                    row[ycol] = int(y_row[j])
                chunk_rows_out.append(row)

        chunk_df = pd.DataFrame(chunk_rows_out)

        if out_csv is not None:
            chunk_df.to_csv(out_csv, mode=write_mode, header=header, index=False, encoding="utf-8")
            write_mode, header = "a", False
        else:
            collector.append(chunk_df)

    if collector is not None:
        return pd.concat(collector, ignore_index=True)
    return None


# -----------------------------
# High-level convenience APIs
# -----------------------------
def classify_and_save(
    texts: Union[str, Sequence[str]],
    out_csv: str = "predicoes.csv",
    model_dir: str = MODEL_DIR,
) -> pd.DataFrame:
    """
    Convenience wrapper for small inputs (list/str). Saves to CSV and returns the DataFrame.
    """
    model, tokenizer, label_names = load_model_and_tokenizer(model_dir)
    thresholds = load_thresholds_or_default(
        os.path.join(model_dir, "thresholds.json"),
        num_labels=len(label_names),
        default=0.5,
    )

    # Print class names for visibility
    print("Class names:", label_names)

    df = predict_multilabel_texts(texts, model, tokenizer, label_names, thresholds)
    df.to_csv(out_csv, index=False, encoding="utf-8")
    return df


def classify_dataframe_and_save(
    df: pd.DataFrame,
    text_col: str,
    out_csv: str,
    model_dir: str = MODEL_DIR,
    chunk_rows: int = 200_000,
    batch_size: int = BATCH_SIZE,
) -> None:
    """
    Scalable wrapper for very large datasets in a DataFrame.
    Streams through the DataFrame in chunks and writes to CSV incrementally.
    Suitable for millions to tens of millions of comments (I/O bound).
    """
    model, tokenizer, label_names = load_model_and_tokenizer(model_dir)
    thresholds = load_thresholds_or_default(
        os.path.join(model_dir, "thresholds.json"),
        num_labels=len(label_names),
        default=0.5,
    )

    print("Class names:", label_names)

    predict_multilabel_dataframe(
        df=df,
        text_col=text_col,
        model=model,
        tokenizer=tokenizer,
        label_names=label_names,
        thresholds=thresholds,
        out_csv=out_csv,
        batch_size=batch_size,
        chunk_rows=chunk_rows,
        write_mode="w",
        header=True,
    )


# -----------------------------
# Examples
# -----------------------------
if __name__ == "__main__":
    examples = [
        "Comecei a usar dura faz 3 semanas e percebi que minha libido caiu.",
        "Meu amigo disse que só teve mais suor à noite depois de usar.",
        "Tô pensando em começar, mas tenho medo de cair cabelo.",
        "Já usei oxan e não senti nada demais.",
        "Depois da aplicação fiquei com dor forte no local.",
        "Não usei nada ainda, mas fico inseguro com esses relatos.",
        "Com 22 anos usei stano e enchi de espinhas.",
        "A disposição aumentou muito depois que iniciei.",
        "Usei 4 semanas e meu cabelo começou a cair.",
        "Nunca tomei, mas penso em resolver minha falta de energia assim.",
        "Fiz TRT por 6 meses, controlei hematócrito e deu tudo certo.",
        "Meu parceiro ficou muito agressivo quando começou o ciclo.",
        "Depois da primeira aplicação já senti pump absurdo.",
        "Usei trembo e tive insônia todas as noites.",
        "Minha prima disse que um colega dela desenvolveu gineco.",
        "Já fiz vários ciclos e nunca tive problema.",
        "No começo tive azia e dor de estômago, depois passou.",
        "Um amigo relatou que ficou ansioso demais usando.",
        "Depois do ciclo perdi tudo e fiquei desanimado.",
        "Ainda não usei, mas quero fazer TPC direitinho quando usar.",
        "Testei oxan e parecia farinha, não senti nada.",
        "Tenho acne mesmo sem usar nada, imagina quem usa.",
        "Com enantato meu sono piorou bastante.",
        "Minha irmã contou que o namorado dela ficou impotente depois.",
        "Fiz 2 ciclos e só dor de cabeça de vez em quando.",
        "Vi muitos casos de depressão depois do uso.",
        "No dia da aplicação tive febre e calafrio.",
        "Tenho medo de prejudicar meu fígado com isso.",
        "Fiz um ciclo leve e não tive nenhum colateral.",
        "Desisti quando vi relatos de muita retenção de líquido."
    ]

    df = classify_and_save(examples, out_csv="predicoes_exemplos.csv", model_dir=MODEL_DIR)

    print("\n=== Results (with per-label thresholds) ===")
    model, _, label_names = load_model_and_tokenizer(MODEL_DIR)
    thresholds = load_thresholds_or_default(
        os.path.join(MODEL_DIR, "thresholds.json"),
        num_labels=len(label_names),
        default=0.5,
    )
    print("Class names:", label_names)
    print("Per-label thresholds:", {name: float(t) for name, t in zip(label_names, thresholds)})

    for _, row in df.iterrows():
        print("\nText:", row["text"])
        for name in label_names:  # use the same label_names to match columns
            pcol = f"probs_{name}".replace(" ", "_").replace("/", "_")
            ycol = f"pred_{name}".replace(" ", "_").replace("/", "_")
            print(f"  {name:>30s} | prob = {row[pcol]:.4f} | class = {row[ycol]}")
    print("\nSaved file: predicoes_exemplos.csv")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class names: ['Autorrelato', 'Sintomas / Efeitos Colaterais']

=== Results (with per-label thresholds) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class names: ['Autorrelato', 'Sintomas / Efeitos Colaterais']
Per-label thresholds: {'Autorrelato': 0.2107662856578827, 'Sintomas / Efeitos Colaterais': 0.66888028383255}

Text: Comecei a usar dura faz 3 semanas e percebi que minha libido caiu.
                     Autorrelato | prob = 0.9971 | class = 1
   Sintomas / Efeitos Colaterais | prob = 0.9849 | class = 1

Text: Meu amigo disse que só teve mais suor à noite depois de usar.
                     Autorrelato | prob = 0.0060 | class = 0
   Sintomas / Efeitos Colaterais | prob = 0.7852 | class = 1

Text: Tô pensando em começar, mas tenho medo de cair cabelo.
                     Autorrelato | prob = 0.0000 | class = 0
   Sintomas / Efeitos Colaterais | prob = 0.6558 | class = 0

Text: Já usei oxan e não senti nada demais.
                     Autorrelato | prob = 0.5190 | class = 1
   Sintomas / Efeitos Colaterais | prob = 0.0303 | class = 0

Text: Depois da aplicação fiquei com dor forte no local.
                     Autorrelato 

# Dados completos

In [ ]:
# -*- coding: utf-8 -*-
"""
Batch multilabel inference on a large CSV with progress (tqdm) and resume.
- Reads input CSV in chunks from `prediction/final_comments_match_cleaned.csv` (text column: 'comment')
- Keeps all original columns and appends 4 prediction columns:
    probs_Autorrelato, pred_Autorrelato,
    probs_Sintomas_Efeitos_Colaterais, pred_Sintomas_Efeitos_Colaterais
- Applies per-label thresholds loaded from ./saved_model/thresholds.json (fallback 0.5)
- 4-bit NF4 + FP16 + optional PEFT adapter
- Resume-safe: writes a small state file to continue after power loss
"""

import os, json, tempfile
os.environ["TOKENIZERS_PARALLELISM"] = "true"
from typing import List
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

# -----------------------------
# Paths & settings
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128
BATCH_SIZE = 512
CHUNK_ROWS = 5000



MODEL_DIR  = "./saved_model"  # ou "./saved_model_merged" se você tiver mesclado o LoRA
INPUT_FILE = "prediction/final_comments_match_cleaned.csv"
OUTPUT_FILE = "prediction/final_comments_match_cleaned_pred.csv"
STATE_FILE  = OUTPUT_FILE + ".state.json"
TEXT_COL = "comment"

LABEL_NAMES = ["Autorrelato", "Sintomas / Efeitos Colaterais"]

# -----------------------------
# Helpers
# -----------------------------
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def _ensure_padding(tokenizer, model):
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is None:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            model.resize_token_embeddings(len(tokenizer))
        else:
            tokenizer.pad_token = tokenizer.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    tokenizer.padding_side = "right"

def load_thresholds(path, num_labels, default=0.5):
    try:
        with open(path, "r", encoding="utf-8") as f:
            ths = np.array(json.load(f)["thresholds"], dtype=float)
        if ths.shape[0] != num_labels:
            ths = np.full(num_labels, default, dtype=float)
    except Exception:
        ths = np.full(num_labels, default, dtype=float)
    return ths

def load_model_and_tokenizer(model_dir=MODEL_DIR):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    compute_dtype = torch.float16
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype
    )

    adapter_cfg = os.path.join(model_dir, "adapter_config.json")
    if os.path.exists(adapter_cfg):
        with open(adapter_cfg, "r", encoding="utf-8") as f:
            base = json.load(f).get("base_model_name_or_path")
        base_model = AutoModelForSequenceClassification.from_pretrained(
            base, quantization_config=bnb, device_map="auto", torch_dtype=compute_dtype
        )
        model = PeftModel.from_pretrained(base_model, model_dir)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir, quantization_config=bnb, device_map="auto", torch_dtype=compute_dtype
        )

    _ensure_padding(tokenizer, model)
    model.eval()

    if len(LABEL_NAMES) != model.config.num_labels:
        raise ValueError("Mismatch between LABEL_NAMES and model.config.num_labels")

    # Optional: make names explicit in config (helps logs/consistency)
    model.config.id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
    model.config.label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

    return model, tokenizer

def _count_rows_fast(csv_path: str) -> int:
    """Low-memory row count (header excluded)."""
    # Robust count without loading file into memory
    with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
        total = sum(1 for _ in f)
    return max(total - 1, 0)

# -----------------------------
# Core prediction 
# -----------------------------
@torch.no_grad()
def _predict_batch(texts, model, tokenizer, thresholds):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    logits_t = model(**inputs).logits.detach()          # [B, L], torch.Tensor
    probs_t  = torch.sigmoid(logits_t)                  # [B, L], estável
    probs = probs_t.cpu().numpy().astype("float32")     # [B, L]
    preds = (probs >= thresholds.reshape(1, -1)).astype("int32")  # [B, L]
    return probs, preds


# -----------------------------
# Main
# -----------------------------
def classify_file(
    input_csv=INPUT_FILE,
    output_csv=OUTPUT_FILE,
    state_file=STATE_FILE,
    text_col=TEXT_COL,
    chunk_rows=CHUNK_ROWS,
    batch_size=BATCH_SIZE,
):
    # Load model/tokenizer/thresholds (no tuning here; just apply saved thresholds)
    model, tokenizer = load_model_and_tokenizer(MODEL_DIR)
    try:
        model.config.attn_implementation = "flash_attention_2"
    except Exception:
        pass

    thresholds = load_thresholds(os.path.join(MODEL_DIR, "thresholds.json"), len(LABEL_NAMES), default=0.5)

    # Resume
    next_chunk = 0
    if os.path.exists(state_file):
        try:
            with open(state_file, "r", encoding="utf-8") as f:
                next_chunk = int(json.load(f).get("next_chunk", 0))
            print(f"[RESUME] Continuing from chunk {next_chunk}")
        except Exception as e:
            print(f"[RESUME] Could not read state file ({e}); starting from 0")

    # Row count just for tqdm ETA (optional but nice)
    total_rows = _count_rows_fast(input_csv)
    total_chunks = (total_rows + chunk_rows - 1) // chunk_rows if total_rows > 0 else None

    # Stream the CSV
    reader = pd.read_csv(input_csv, chunksize=chunk_rows)
    for chunk_idx, df_chunk in enumerate(tqdm(reader, total=total_chunks, desc="Processing", unit="chunk")):
        if chunk_idx < next_chunk:
            continue

        if text_col not in df_chunk.columns:
            raise KeyError(f"Column '{text_col}' not found in CSV chunk.")

        texts = df_chunk[text_col].astype(str).tolist()

        # Predict in mini-batches
        probs_all, preds_all = [], []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            probs, preds = _predict_batch(batch, model, tokenizer, thresholds)
            probs_all.append(probs); preds_all.append(preds)

        probs_all = np.vstack(probs_all)
        preds_all = np.vstack(preds_all)

        # Append 4 new columns (keep all existing ones)
        # Safer col names (no spaces/slashes)
        p_aut  = "probs_Autorrelato"
        y_aut  = "pred_Autorrelato"
        p_sint = "probs_Sintomas_Efeitos_Colaterais"
        y_sint = "pred_Sintomas_Efeitos_Colaterais"

        df_chunk[p_aut]  = probs_all[:, 0]
        df_chunk[y_aut]  = preds_all[:, 0].astype(int)
        df_chunk[p_sint] = probs_all[:, 1]
        df_chunk[y_sint] = preds_all[:, 1].astype(int)

        # Atomic-ish write per chunk: write to tmp, then append/move
        mode, header = ("w", True) if (chunk_idx == 0 and next_chunk == 0 and not os.path.exists(output_csv)) else ("a", False)
        dir_name = os.path.dirname(os.path.abspath(output_csv)) or "."
        fd, tmp_path = tempfile.mkstemp(suffix=".part", dir=dir_name)
        os.close(fd)
        try:
            df_chunk.to_csv(tmp_path, mode="w", header=header, index=False, encoding="utf-8")
            if mode == "a" and os.path.exists(output_csv):
                # Append contents of temp to output (skip header if present)
                with open(output_csv, "a", encoding="utf-8") as fout, open(tmp_path, "r", encoding="utf-8") as fin:
                    if header:
                        next(fin, None)
                    for line in fin:
                        fout.write(line)
                os.remove(tmp_path)
            else:
                os.replace(tmp_path, output_csv)

            # Update state only after successful write
            with open(state_file, "w", encoding="utf-8") as f:
                json.dump({"next_chunk": chunk_idx + 1}, f)
        except Exception as e:
            print(f"[WRITE] Error at chunk {chunk_idx}: {e}. Temp kept at {tmp_path}")
            raise

    # Done → remove state
    if os.path.exists(state_file):
        os.remove(state_file)
    print(f"✅ Done. Output saved to: {output_csv}")

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    # Make sure the folder 'prediction/' exists
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    classify_file()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[RESUME] Continuing from chunk 23


Processing:   0%|          | 0/123 [00:00<?, ?chunk/s]